In [18]:
import numpy as np
import OpenVisus as ov
import matplotlib.pyplot as plt
import os
from pathlib import Path

variable = 'u'
face=2
timestep=1

geos_face_loc=f"https://nsdf-climate3-origin.nationalresearchplatform.org:50098/nasa/nsdf/climate3/dyamond/GEOS/GEOS_{variable.upper()}/{variable.lower()}_face_{face}_depth_52_time_0_10269.idx"

In [19]:
db=ov.LoadDataset(geos_face_loc)
print(f'Dimensions: {db.getLogicBox()[1][0]}*{db.getLogicBox()[1][1]}*{db.getLogicBox()[1][2]}')
print(f'Total Timesteps: {len(db.getTimesteps())}')
print(f'Field: {db.getField().name}')
print('Data Type: float32')

Dimensions: 1440*1440*52
Total Timesteps: 10269
Field: u
Data Type: float32


In [20]:
data=db.read(time=0,quality=-4,z=[50,51]) 
data.shape
print(data)

[[[-1.0508074  -0.9231218  -0.9310564  ... -0.6696428  -0.9437517
   -1.1400408 ]
  [-1.5538591  -1.336574   -1.1747087  ... -0.59920824 -0.80569017
   -0.9051775 ]
  [-1.9996599  -1.9767107  -1.6603044  ... -0.50237596 -0.6209215
   -0.64466417]
  ...
  [ 4.1800275   3.7913556   3.3724103  ...  6.566746    6.1468244
    5.6839337 ]
  [ 4.2581525   3.6390119   2.8860822  ...  6.6800275   6.2601056
    5.8226056 ]
  [ 3.703465    2.971043    4.873387   ...  6.097996    5.8870587
    5.639012  ]]]


In [21]:
db = ov.LoadDataset(geos_face_loc)

print(f'Dimensions: {db.getLogicBox()[1]}')
print(f'Total Timesteps: {len(db.getTimesteps())}')
print(f'Field: {db.getField().name}')

Dimensions: [1440, 1440, 52]
Total Timesteps: 10269
Field: u


In [22]:
logic_box = db.getLogicBox()
dims = logic_box[1]
print("Full spatial dimensions:", dims)

Full spatial dimensions: [1440, 1440, 52]


In [23]:
data = db.read(
    time=0,
    quality=-4
)

In [24]:
print(data.shape)

(52, 360, 360)


In [25]:
import numpy as np
import pandas as pd

Z, Y, X = data.shape

# Create coordinate indices
z_vals, y_vals, x_vals = np.meshgrid(
    np.arange(Z),
    np.arange(Y),
    np.arange(X),
    indexing='ij'
)

# Flatten everything
df = pd.DataFrame({
    'z': z_vals.flatten(),
    'y': y_vals.flatten(),
    'x': x_vals.flatten(),
    'u_value': data.flatten()
})
filepath = Path("csv_files/u_time0_full.csv")
os.makedirs(os.path.dirname(filepath), exist_ok=True)
df.to_csv(filepath, index=False)

print("CSV saved.")

CSV saved.


In [26]:
# If sklearn is available, we'll use it. Otherwise fallback to numpy SVD implementation.
try:
    from sklearn.decomposition import PCA
    _HAVE_SKLEARN = True
except Exception:
    _HAVE_SKLEARN = False
    # We'll implement PCA via SVD below if sklearn isn't present.

# --- assume data is an ndarray with shape (Z, Y, X) ---
Z, Y, X = data.shape

# create output folder
out_dir = Path("csv_files")
out_dir.mkdir(parents=True, exist_ok=True)

# ----------------------------
# Option A: Top-layer collapse
# ----------------------------
top_z = 0  # change this if you want a different depth slice (e.g., -1 for bottom)
u_top = data[top_z, :, :]  # shape (Y, X)

rows = []
for y in range(Y):
    for x in range(X):
        rows.append((y, x, float(u_top[y, x])))

df_top = pd.DataFrame(rows, columns=["y", "x", "u_value"])
top_path = out_dir / "u_time0_toplayer.csv"
df_top.to_csv(top_path, index=False)
print(f"Top-layer CSV saved to: {top_path} (z={top_z})")

# ---------------------------------------
# Option B: Flatten depth using PCA (Z -> 1)
# ---------------------------------------
# Build matrix of shape (n_samples, n_features) = (Y*X, Z)
# Each row = depth vector for a particular (y,x)
data_reshaped = data.reshape(Z, Y * X).T  # shape (Y*X, Z)

if _HAVE_SKLEARN:
    pca = PCA(n_components=1)
    scores = pca.fit_transform(data_reshaped)[:, 0]  # length Y*X
    explained_var_ratio = float(pca.explained_variance_ratio_[0])
else:
    # Fallback PCA via SVD (center data then take first left singular vector)
    # Center
    Xc = data_reshaped - np.mean(data_reshaped, axis=0, keepdims=True)
    # SVD
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    # Project onto first right singular vector (principal direction)
    pc1 = Vt[0]  # shape (Z,)
    scores = Xc.dot(pc1)  # projection scores (Y*X,)
    # estimate explained variance ratio for first PC
    total_var = np.sum(np.var(data_reshaped, axis=0, ddof=0))
    pc1_var = np.var(scores, ddof=0)
    explained_var_ratio = float(pc1_var / total_var) if total_var > 0 else 0.0

# Put scores back into (Y, X)
scores_2d = scores.reshape(Y, X)

# Save CSV with (y, x, pca_value)
rows = []
for y in range(Y):
    for x in range(X):
        rows.append((y, x, float(scores_2d[y, x])))

df_pca = pd.DataFrame(rows, columns=["y", "x", "u_pca1"])
pca_path = out_dir / "u_time0_pca_flatten.csv"
df_pca.to_csv(pca_path, index=False)
print(f"PCA-flatten CSV saved to: {pca_path}")
print(f"Explained variance ratio by first PC: {explained_var_ratio:.4f}")

Top-layer CSV saved to: csv_files\u_time0_toplayer.csv (z=0)
PCA-flatten CSV saved to: csv_files\u_time0_pca_flatten.csv
Explained variance ratio by first PC: 0.7247
